# Silver Customer Implementation

Reads unmerged bronze customers, cleans and validates them, MERGEs valid rows into silver,
writes rejected rows to a quarantine table with a reason, then flags processed bronze rows.

| Outcome | Destination | Bronze `merge_flag` |
|---|---|---|
| Valid | `silver.customer` | `true` |
| Hard reject (bad data that will never become valid) | `silver.quarantine_customer` | `true` |

Customer has no parent table, so there are no soft (retry) rejects.
**Job order:** runs first, in parallel with Silver_Branch. Silver_Policy depends on it.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.customer (
  customer_id       INT,
  first_name        STRING,
  last_name         STRING,
  email             STRING,
  phone             STRING,
  country           STRING,
  city              STRING,
  registration_date TIMESTAMP,
  date_of_birth     TIMESTAMP,
  gender            STRING,
  merged_timestamp  TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/customer';

CREATE TABLE IF NOT EXISTS policysystemdatabricks.silver.quarantine_customer (
  customer_id           INT,
  first_name            STRING,
  last_name             STRING,
  email                 STRING,
  phone                 STRING,
  country               STRING,
  city                  STRING,
  registration_date     TIMESTAMP,
  date_of_birth         TIMESTAMP,
  gender                STRING,
  reject_reason         STRING,
  quarantined_timestamp TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@insurancepolicy1adls.dfs.core.windows.net/quarantine/customer';

In [0]:
from pyspark.sql import functions as F, Window

COLUMNS = ["customer_id", "first_name", "last_name", "email", "phone", "country", "city",
           "registration_date", "date_of_birth", "gender"]

# 1. Read only unmerged bronze customers
customer_df = spark.sql("SELECT * FROM policysystemdatabricks.bronze.customer WHERE merge_flag = false")

# 2. Clean: trim / collapse spaces, lower-case email, default missing descriptive values
clean_df = customer_df.selectExpr(
    "customer_id",
    r"coalesce(nullif(regexp_replace(trim(first_name), '\\s+', ' '), ''), 'Unknown') AS first_name",
    r"coalesce(nullif(regexp_replace(trim(last_name), '\\s+', ' '), ''), 'Unknown') AS last_name",
    "coalesce(nullif(lower(trim(email)), ''), 'admin@azurelib.com') AS email",
    "coalesce(nullif(trim(phone), ''), '0000000000') AS phone",
    r"coalesce(nullif(regexp_replace(trim(country), '\\s+', ' '), ''), 'Unknown') AS country",
    r"coalesce(nullif(regexp_replace(trim(city), '\\s+', ' '), ''), 'Unknown') AS city",
    "registration_date",
    "date_of_birth",
    "initcap(nullif(trim(gender), '')) AS gender",  # no default: only Male / Female pass validation
)

# 3. Deduplicate: one row per customer_id (latest registration_date). Extra sort keys make the pick deterministic.
#    NULL customer_id rows are kept individually so each one is quarantined.
w = Window.partitionBy("customer_id").orderBy(
    F.col("registration_date").desc_nulls_last(), F.col("email"), F.col("phone"))
df_result = (clean_df.withColumn("rn", F.row_number().over(w))
             .filter("rn = 1 OR customer_id IS NULL")
             .drop("rn"))

In [0]:
reject_reason = (
    F.when(F.col("customer_id").isNull(), "null_customer_id")
     .when(F.col("gender").isNull() | ~F.col("gender").isin("Male", "Female"), "invalid_gender")
     .when(F.col("registration_date").isNull() | F.col("date_of_birth").isNull(), "missing_dates")
     .when(F.col("registration_date") <= F.col("date_of_birth"), "registration_not_after_birth")
     .when(~F.col("phone").rlike(r"^[0-9]{10}$"), "invalid_phone")
     .when(~F.col("email").rlike(r"^[^@ ]+@[^@ ]+\.[^@ ]+$"), "invalid_email")
)

classified_df = df_result.withColumn("reject_reason", reject_reason)
classified_df.createOrReplaceTempView("customer_classified")

display(classified_df.groupBy(F.coalesce("reject_reason", F.lit("VALID")).alias("outcome")).count())

outcome,count
null_customer_id,64


In [0]:
display(spark.sql("""
MERGE INTO policysystemdatabricks.silver.customer AS t
USING (SELECT * FROM customer_classified WHERE reject_reason IS NULL) AS s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET
  t.first_name        = s.first_name,
  t.last_name         = s.last_name,
  t.email             = s.email,
  t.phone             = s.phone,
  t.country           = s.country,
  t.city              = s.city,
  t.registration_date = s.registration_date,
  t.date_of_birth     = s.date_of_birth,
  t.gender            = s.gender,
  t.merged_timestamp  = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (customer_id, first_name, last_name, email, phone, country, city, registration_date, date_of_birth, gender, merged_timestamp)
  VALUES (s.customer_id, s.first_name, s.last_name, s.email, s.phone, s.country, s.city, s.registration_date, s.date_of_birth, s.gender, current_timestamp())
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
display(spark.sql(f"""
INSERT INTO policysystemdatabricks.silver.quarantine_customer
SELECT {", ".join(COLUMNS)}, reject_reason, current_timestamp()
FROM customer_classified
WHERE reject_reason IS NOT NULL
"""))

num_affected_rows,num_inserted_rows
64,64


In [0]:
# Valid and quarantined keys are both "processed" (incl. their older duplicates).
display(spark.sql("""
MERGE INTO policysystemdatabricks.bronze.customer AS b
USING (SELECT DISTINCT customer_id FROM customer_classified WHERE customer_id IS NOT NULL) AS s
ON b.customer_id = s.customer_id AND b.merge_flag = false
WHEN MATCHED THEN UPDATE SET b.merge_flag = true
"""))

# NULL keys cannot be matched in a MERGE; they are always quarantined above, so flag them directly.
display(spark.sql("""
UPDATE policysystemdatabricks.bronze.customer
SET merge_flag = true
WHERE merge_flag = false AND customer_id IS NULL
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


num_affected_rows
64


In [0]:
%sql
SELECT (SELECT count(*) FROM policysystemdatabricks.bronze.customer WHERE merge_flag = false) AS bronze_unmerged_rows,  -- should be 0
       (SELECT count(*) FROM policysystemdatabricks.silver.quarantine_customer) AS quarantined_rows_total,
       count(*)                                  AS silver_rows,
       count(DISTINCT customer_id)               AS silver_distinct_customers,
       count_if(first_name = 'Unknown' OR last_name = 'Unknown') AS default_names,
       count_if(email = 'admin@azurelib.com')    AS default_emails,
       count_if(phone = '0000000000')            AS default_phones,
       count_if(country = 'Unknown' OR city = 'Unknown') AS default_locations,
       count_if(merged_timestamp IS NULL)        AS rows_missing_merged_timestamp,  -- should be 0
       count_if(gender NOT IN ('Male', 'Female') OR gender IS NULL) AS invalid_gender,  -- 0 after the one-time fixes
       count_if(NOT (registration_date > date_of_birth) OR registration_date IS NULL OR date_of_birth IS NULL) AS dob_outliers  -- 0 after the one-time fixes
FROM policysystemdatabricks.silver.customer

bronze_unmerged_rows,quarantined_rows_total,silver_rows,silver_distinct_customers,default_names,default_emails,default_phones,default_locations,rows_missing_merged_timestamp,invalid_gender,dob_outliers
0,64,968,968,85,53,0,85,0,65,512
